<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/notebooks/06_gcn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/harsh3-web/Ecommerce-Recommender.git
%cd Ecommerce-Recommender
# torch_geometric is needed because we'll also test the OLD functions from model.py
!pip install -q sentence-transformers torch_geometric

Cloning into 'Ecommerce-Recommender'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 28 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 58.15 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/Ecommerce-Recommender
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/Ecommerce-Recommender-data/*.csv .
!ls *.csv

Mounted at /content/drive
browsing_history_expanded.csv  products_expanded.csv
ground_truth_tastes.csv        purchases_expanded.csv
product_images_expanded.csv    users_expanded.csv


In [ ]:
import numpy as np
import pandas as pd

products  = pd.read_csv("products_expanded.csv")
users     = pd.read_csv("users_expanded.csv")
# parse_dates -> timestamps become real dates, so we can find each user's LAST purchase
purchases = pd.read_csv("purchases_expanded.csv", parse_dates=["timestamp"])
browsing  = pd.read_csv("browsing_history_expanded.csv", parse_dates=["timestamp"])
n_users, n_products = len(users), len(products)

In [ ]:
# sort by time, then take the LAST row per user = their most recent purchase
last = purchases.sort_values("timestamp").groupby("user_id").tail(1)

n_buys = purchases.groupby("user_id").size()          # purchases per user
eligible = n_buys[n_buys >= 2].index                  # need >= 1 purchase LEFT for training
test_items = last[last["user_id"].isin(eligible)].set_index("user_id")["product_id"]
print(len(test_items), "users will be evaluated")     # test_items[u] = the hidden product for user u

455 users will be evaluated


In [ ]:
def drop_test(df):
    """Remove every row where a user interacts with THEIR OWN hidden test product."""
    hidden = df["user_id"].map(test_items)     # hidden product for each row's user (NaN if not evaluated)
    return df[df["product_id"] != hidden].copy()

train_purchases = drop_test(purchases)
train_browsing  = drop_test(browsing)
print(f"Purchases: {len(purchases)} -> {len(train_purchases)} | Browsing: {len(browsing)} -> {len(train_browsing)}")

Purchases: 1901 -> 1446 | Browsing: 6269 -> 5814


In [ ]:
rng = np.random.default_rng(42)
eval_users = test_items.index.to_numpy()
rng.shuffle(eval_users)                       # random order (fixed by the seed)
half = len(eval_users) // 2
val_users, test_users = eval_users[:half], eval_users[half:]
print(len(val_users), "validation users |", len(test_users), "test users")

227 validation users | 228 test users


In [ ]:
def hit_at_k(ranked_ids, true_id, k=10):
    """1 if the hidden product appears anywhere in the top k, else 0."""
    return float(true_id in ranked_ids[:k])

def ndcg_at_k(ranked_ids, true_id, k=10):
    """Like hit rate, but rewards ranking the hidden product HIGHER."""
    top = list(ranked_ids[:k])
    if true_id not in top:
        return 0.0
    rank = top.index(true_id) + 1             # position 1, 2, ... 10
    return 1 / np.log2(rank + 1)              # rank 1 -> 1.0, rank 2 -> 0.63, rank 10 -> 0.29

In [ ]:
def evaluate(recommend_fn, user_ids, k=10):
    """Average HR@k and NDCG@k over users. recommend_fn(u) must return a table sorted best-first."""
    hits, ndcgs = [], []
    for u in user_ids:
        ranked = recommend_fn(u)["product_id"].tolist()
        hits.append(hit_at_k(ranked, test_items[u], k))
        ndcgs.append(ndcg_at_k(ranked, test_items[u], k))
    return {"HR@10": np.mean(hits), "NDCG@10": np.mean(ndcgs)}

In [ ]:
from scipy.sparse import csr_matrix
BROWSE_W, BUY_W = 1.0, 4.0
b = train_browsing[["user_id", "product_id"]].assign(weight=BROWSE_W)
p = train_purchases[["user_id", "product_id"]].assign(weight=BUY_W)
inter = pd.concat([b, p]).groupby(["user_id", "product_id"], as_index=False)["weight"].sum()

R = csr_matrix((inter["weight"].values, (inter["user_id"].values - 1, inter["product_id"].values - 1)),
               shape=(n_users, n_products))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
item_sim = cosine_similarity(R.T)        # 200 x 200, built from TRAINING data only
np.fill_diagonal(item_sim, 0)

# Dictionary: user_id -> set of products they already touched (in training). Fast to look up.
seen = inter.groupby("user_id")["product_id"].apply(set).to_dict()

In [ ]:
def to_recs(scores, uid, source):
    """Array of 200 scores (product_id order) -> ranked table without already-seen products."""
    recs = products.copy()
    recs["score"], recs["source"] = scores, source
    recs = recs[~recs["product_id"].isin(seen.get(uid, set()))]   # .get -> empty set if user unknown
    return recs.sort_values("score", ascending=False)

In [ ]:
def random_recs(uid):
    return to_recs(rng.random(n_products), uid, "Random")       # random scores -> random ranking

# How often each product was bought in TRAINING; reindex fills never-bought products with 0
pop = train_purchases["product_id"].value_counts().reindex(products["product_id"], fill_value=0).values
def popular(uid):
    return to_recs(pop.astype(float), uid, "Popular")            # same bestseller list for everyone

In [ ]:
def item_cf(uid):
    user_vec = R[uid - 1].toarray().ravel()      # this user's training weights
    return to_recs(user_vec @ item_sim, uid, "Item-CF")

In [ ]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
texts = (products["category"] + ". " + products["product_name"] + ". " + products["description"]).tolist()
item_emb = encoder.encode(texts, normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def content_emb(uid):
    mine = inter[inter["user_id"] == uid]                          # TRAINING interactions only
    w = mine["weight"].values[:, None]
    profile = (w * item_emb[mine["product_id"].values - 1]).sum(axis=0) / w.sum()
    profile = profile / np.linalg.norm(profile)
    return to_recs(item_emb @ profile, uid, "Content-Emb")

In [ ]:
def minmax(s):
    spread = s.max() - s.min()
    return (s - s.min()) / spread if spread > 0 else s * 0

def hybrid_v2(uid, alpha=0.5):
    cf = minmax(item_cf(uid).set_index("product_id")["score"])
    cb = minmax(content_emb(uid).set_index("product_id")["score"])
    combined = (alpha * cf + (1 - alpha) * cb).sort_values(ascending=False)
    return products.set_index("product_id").loc[combined.index].reset_index().assign(score=combined.values)

In [ ]:
##Load the image embeddings

DATA = "/content/drive/MyDrive/Ecommerce-Recommender-data"
!cp "{DATA}/image_emb.npy" .
image_emb = np.load("image_emb.npy")      # (200, 2048), saved on Day 4, rows in product_id order
print(image_emb.shape, "| text:", item_emb.shape)   # item_emb = text embeddings from the copied cells

(200, 2048) | text: (200, 384)


In [ ]:
##The user × product purchase matrix (training data only)
from scipy.sparse import csr_matrix

# B[u, p] = 1 if user u bought product p (TRAINING purchases only -> no leakage of test items)
B = csr_matrix((np.ones(len(train_purchases)),
                (train_purchases["user_id"].values - 1, train_purchases["product_id"].values - 1)),
               shape=(n_users, n_products))
B.data[:] = 1.0                       # make sure every entry is exactly 1, even if repeated
print(B.shape, "| purchases:", B.nnz)

(500, 200) | purchases: 1446


In [ ]:
##Co-purchase counts and the adjacency matrix
# B.T @ B -> C[i, j] = number of users who bought BOTH product i and product j
C = (B.T @ B).toarray()
np.fill_diagonal(C, 0)                # a product is not its own neighbour

A = (C > 0).astype(float)             # adjacency matrix: 1 if co-purchased at least once, else 0
print("Edges:", int(A.sum() / 2))     # /2 because A is symmetric (i-j and j-i are the same edge)

Edges: 1753


In [ ]:
##Graph statistics
degree = A.sum(axis=1)                          # number of neighbours of each product
print(f"Average neighbours per product: {degree.mean():.1f}")
print(f"Max neighbours: {int(degree.max())} | Min: {int(degree.min())}")
print(f"Isolated products (no edges): {(degree == 0).sum()}")    # cold-start in graph terms
print(f"Graph density: {A.sum() / (n_products * (n_products - 1)):.1%}")  # share of possible edges present

Average neighbours per product: 17.5
Max neighbours: 79 | Min: 0
Isolated products (no edges): 7
Graph density: 8.8%


In [ ]:
##Convert to PyTorch Geometric's format
import torch

src, dst = np.nonzero(A)              # row and column positions of every 1 in A
edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
print(edge_index.shape)               # (2, number_of_directed_edges)
print(edge_index[:, :5])              # first 5 edges: top row = from, bottom row = to

torch.Size([2, 3506])
tensor([[0, 0, 0, 0, 0],
        [1, 2, 5, 7, 9]])


In [ ]:
#Build the normalised adjacency Â
A_hat = A + np.eye(n_products)              # add SELF-LOOPS: each product also counts itself
deg_hat = A_hat.sum(axis=1)                 # degree including the self-loop (always >= 1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(deg_hat))   # diagonal matrix of 1/sqrt(degree)
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt     # symmetric normalisation: D^-1/2 (A+I) D^-1/2
print(A_norm.shape, "| row 0 sums to", round(A_norm[0].sum(), 3))

(200, 200) | row 0 sums to 0.975


In [ ]:
##One propagation step by hand
X = item_emb[:, :4]                  # take just 4 text features so the numbers are easy to read
H1 = A_norm @ X                      # ONE GCN step (W = identity): mix each product with its neighbours

pid = 1
print("Neighbours of product 1:", (np.nonzero(A[pid - 1])[0] + 1).tolist())
print("Before:", X[pid - 1].round(3))
print("After: ", H1[pid - 1].round(3))   # now a weighted blend of itself + its neighbours

Neighbours of product 1: [2, 3, 6, 8, 10, 13, 15, 16, 19, 22, 27, 34, 40, 47, 55, 60, 63, 129, 154, 159, 164, 176, 180, 193]
Before: [-0.022  0.035  0.011  0.078]
After:  [-0.012  0.044  0.021  0.062]


In [ ]:
##Prove that GCNConv does exactly this
from torch_geometric.nn import GCNConv

conv = GCNConv(in_channels=4, out_channels=4, bias=False)   # one GCN layer: 4 inputs -> 4 outputs
with torch.no_grad():
    conv.lin.weight.copy_(torch.eye(4))        # set W = identity matrix -> "no transformation"
out = conv(torch.tensor(X, dtype=torch.float), edge_index).detach().numpy()

print("Same as my manual version?", np.allclose(out, H1, atol=1e-5))   # should print True

Same as my manual version? True


In [ ]:
##Helpers: normalise rows and propagate K steps
def l2(x):
    """Scale every row to length 1 (so dot product = cosine similarity)."""
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def propagate(X, k):
    """Apply k GCN steps (W = identity): each step mixes in one more 'hop' of neighbours."""
    H = X.copy()
    for _ in range(k):
        H = A_norm @ H
    return l2(H)

In [ ]:
##A recommender that works with any embedding
def make_profile_recommender(emb, name):
    """Build a 'taste vector' recommender for any product embedding matrix."""
    def recommend(uid):
        mine = inter[inter["user_id"] == uid]                   # user's TRAINING interactions
        w = mine["weight"].values[:, None]                      # browse = 1, purchase = 5
        profile = (w * emb[mine["product_id"].values - 1]).sum(axis=0) / w.sum()
        return to_recs(emb @ (profile / np.linalg.norm(profile)), uid, name)
    return recommend                                            # returns a FUNCTION

In [ ]:
##Which features and how many hops? (validation only)
features = {"Text": item_emb, "Image": image_emb,
            "Text+Image": l2(np.hstack([item_emb, image_emb]))}   # both, equally weighted

rows = []
for fname, X in features.items():
    for k in [0, 1, 2, 3]:
        rec = make_profile_recommender(propagate(X, k), f"{fname} K={k}")
        rows.append({"Features": fname, "K": k, **evaluate(rec, val_users)})
sweep = pd.DataFrame(rows)
sweep.pivot(index="Features", columns="K", values="NDCG@10").round(3)   # table: features x hops

K,0,1,2,3
Features,,,,
Image,0.058,0.174,0.132,0.115
Text,0.104,0.174,0.148,0.119
Text+Image,0.094,0.176,0.137,0.120


In [ ]:
##Pick the best configuration (on validation)
best = sweep.loc[sweep["NDCG@10"].idxmax()]            # row with the highest validation NDCG
print("Best:", best["Features"], "with K =", int(best["K"]), "| val NDCG:", round(best["NDCG@10"], 3))

graph_emb = propagate(features[best["Features"]], int(best["K"]))
graph_rec = make_profile_recommender(graph_emb, "Graph-Content")

Best: Text+Image with K = 1 | val NDCG: 0.176


In [ ]:
##Hybrid of item-CF and graph-content
def hybrid_graph(uid, alpha):
    cf = minmax(item_cf(uid).set_index("product_id")["score"])
    gc = minmax(graph_rec(uid).set_index("product_id")["score"])
    combined = (alpha * cf + (1 - alpha) * gc).sort_values(ascending=False)
    return products.set_index("product_id").loc[combined.index].reset_index().assign(score=combined.values)

for a in [0.25, 0.5, 0.75]:     # tune alpha on VALIDATION users
    print(f"alpha={a}: val NDCG = {evaluate(lambda u: hybrid_graph(u, a), val_users)['NDCG@10']:.3f}")

alpha=0.25: val NDCG = 0.192
alpha=0.5: val NDCG = 0.181
alpha=0.75: val NDCG = 0.178


In [ ]:
##Final comparison on TEST users
BEST_A = 0.25     # <- replace with your best alpha from Cell 13

final = {
    "Popular": popular, "Item-CF": item_cf,
    "Hybrid Day 3 (a=0.75)": lambda u: hybrid_v2(u, alpha=0.75),
    "Graph-Content": graph_rec,
    f"Hybrid-Graph (a={BEST_A})": lambda u: hybrid_graph(u, BEST_A),
}
pd.DataFrame([{"Method": n, **evaluate(f, test_users)} for n, f in final.items()]).set_index("Method").round(3)

,HR@10,NDCG@10
Method,,
Popular,0.211,0.108
Item-CF,0.373,0.196
Hybrid Day 3 (a=0.75),0.364,0.197
Graph-Content,0.276,0.147
Hybrid-Graph (a=0.25),0.325,0.189


In [ ]:
##Save the graph for Day 6
np.save("edge_index.npy", edge_index.numpy())     # the training co-purchase graph
!cp edge_index.npy "{DATA}/"
print("Saved edge_index:", edge_index.shape)

Saved edge_index: torch.Size([2, 3506])


##  - GCN on co-purchase graph
Graph (training purchases only): 200 nodes, 1,753 edges, avg 17.5 neighbours,
max 79 (bestseller hub), 7 isolated products, density 8.8%
Manual D^-1/2 (A+I) D^-1/2 propagation == PyG GCNConv (W = identity): True

Validation NDCG@10 by hops K:
| Features   | K=0   | K=1   | K=2   | K=3   |
|------------|-------|-------|-------|-------|
| Image      | 0.058 | 0.174 | 0.132 | 0.115 |
| Text       | 0.104 | 0.174 | 0.148 | 0.119 |
| Text+Image | 0.094 | 0.176 | 0.137 | 0.120 |
-> 1 hop best; 2+ hops = over-smoothing. Image gained most (3x).

Test results:
| Method                  | HR@10 | NDCG@10 |
|-------------------------|-------|---------|
| Popular                 | 0.211 | 0.108   |
| Item-CF                 | 0.373 | 0.196   |
| Hybrid Day 3 (a=0.75)   | 0.364 | 0.197   |  <- final model
| Graph-Content (K=1)     | 0.276 | 0.147   |
| Hybrid-Graph (a=0.25)   | 0.325 | 0.189   |

Takeaways:
- GCN lifts content-based NDCG +43% (0.103 -> 0.147), above Popular
- No gain on top of item-CF: both use the same co-purchase signal
- Validated alpha (0.25) didn't transfer: val differences (~0.01) < noise;
  would use cross-validation with more data